# Bayer Leverkusen 2023/24 — data preparation

This notebook pulls StatsBomb open data for Bayer Leverkusen's 34 Bundesliga matches of the 2023/24 season
and exports the clean tables used by the Power BI report (`Data/*.csv`, `;`-separated, UTF-8 with BOM).

Run it top to bottom (*Restart & Run All*): every CSV in `Data/` is regenerated, except
`player_extra_info.csv`, which holds the hand-curated player info (general position, photo URL).

In [1]:
import warnings
from pathlib import Path

import pandas as pd
from statsbombpy import sb

warnings.filterwarnings("ignore", module="statsbombpy")  # "no credentials" notice: open data only

TEAM = "Bayer Leverkusen"
COMPETITION_ID = 9   # 1. Bundesliga
SEASON_ID = 281      # 2023/2024
DATA_DIR = Path("Data")
DATA_DIR.mkdir(exist_ok=True)

def export(df, filename):
    """Write a table in the format expected by the Power BI report."""
    df.to_csv(DATA_DIR / filename, index=False, encoding="utf-8-sig", sep=";", lineterminator="\r\n")
    print(f"{filename}: {len(df)} rows")

## Matches

In [2]:
matches_df = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)

bayer_matches_df = (
    matches_df[(matches_df["home_team"] == TEAM) | (matches_df["away_team"] == TEAM)]
    [["match_id", "competition", "match_week", "season", "match_date", "stadium",
      "home_team", "away_team", "home_score", "away_score"]]
    .sort_values("match_week")
    .reset_index(drop=True)
)
bayer_match_ids = bayer_matches_df["match_id"].tolist()

export(bayer_matches_df, "bayer_matches.csv")
bayer_matches_df.head()

bayer_matches.csv: 34 rows


,match_id,competition,match_week,season,match_date,stadium,home_team,away_team,home_score,away_score
0,3895052,Germany - 1. Bundesliga,1,2023/2024,2023-08-19,BayArena,Bayer Leverkusen,RB Leipzig,3,2
1,3895060,Germany - 1. Bundesliga,2,2023/2024,2023-08-26,Stadion im BORUSSIA-PARK,Borussia Mönchengladbach,Bayer Leverkusen,0,3
2,3895067,Germany - 1. Bundesliga,3,2023/2024,2023-09-02,BayArena,Bayer Leverkusen,Darmstadt 98,5,1
3,3895074,Germany - 1. Bundesliga,4,2023/2024,2023-09-15,Allianz Arena,Bayern Munich,Bayer Leverkusen,2,2
4,3895086,Germany - 1. Bundesliga,5,2023/2024,2023-09-24,BayArena,Bayer Leverkusen,FC Heidenheim,4,1


## Events and lineups

Only Bayer's 34 matches are downloaded (rather than the whole league), which is much faster.

In [3]:
events_df = pd.concat([sb.events(match_id=m) for m in bayer_match_ids], ignore_index=True)
lineups_df = pd.concat(
    [sb.lineups(match_id=m)[TEAM].assign(match_id=m) for m in bayer_match_ids],
    ignore_index=True,
)

def split_xy(series):
    """Split a column of [x, y] (or [x, y, z]) lists into two numeric columns."""
    x = series.apply(lambda loc: loc[0] if isinstance(loc, list) else None)
    y = series.apply(lambda loc: loc[1] if isinstance(loc, list) else None)
    return pd.to_numeric(x), pd.to_numeric(y)

events_df["location_x"], events_df["location_y"] = split_xy(events_df["location"])
events_df["pass_end_location_x"], events_df["pass_end_location_y"] = split_xy(events_df["pass_end_location"])
events_df["shot_end_location_x"], events_df["shot_end_location_y"] = split_xy(events_df["shot_end_location"])

# IDs as integers (3500, not 3500.0) so they match the lineup IDs in Tableau / Power BI relationships
for col in ["player_id", "pass_recipient_id"]:
    events_df[col] = events_df[col].astype("Int64")

bayer_events_df = events_df[events_df["team"] == TEAM]
print(f"{len(events_df)} events, {len(bayer_events_df)} by {TEAM}")

137765 events, 81440 by Bayer Leverkusen


## Squad

General position and photo URL are not in StatsBomb: they live in `Data/player_extra_info.csv`
(previously added by hand in Excel) and are merged here, so the export is reproducible.

In [4]:
name_corrections = {
    "Alejandro Grimaldo García": "Alejandro Grimaldo",
    "Borja Iglesias Quintas": "Borja Iglesias",
    "Exequiel Alejandro Palacios": "Exequiel Palacios",
    "Edmond Fayçal Tapsoba": "Edmond Tapsoba",
    "Victor Okoh Boniface": "Victor Boniface",
    "Piero Martín Hincapié Reyna": "Piero Hincapié",
    "Arthur Augusto de Matos Soares": "Arthur",
    "Gustavo Adolfo Puerta Molano": "Gustavo Puerta",
}

def jersey_name(common_name):
    """Surname in capitals (second word, or the only word for mononyms like 'Arthur')."""
    parts = common_name.split()
    return (parts[1] if len(parts) > 1 else parts[0]).upper()

# Keep only players who appear in at least one Bayer event (i.e. actually played)
players_with_events = bayer_events_df["player_id"].dropna().unique()

bayer_squad_df = (
    lineups_df.drop_duplicates("player_id")
    [["player_id", "player_name", "jersey_number", "country"]]
    .loc[lambda d: d["player_id"].isin(players_with_events)]
    .reset_index(drop=True)
)
bayer_squad_df["common_name"] = bayer_squad_df["player_name"].replace(name_corrections)
bayer_squad_df["jersey_name"] = bayer_squad_df["common_name"].apply(jersey_name)

extra_info = pd.read_csv(DATA_DIR / "player_extra_info.csv", sep=";", encoding="utf-8-sig")
bayer_squad_df = bayer_squad_df.merge(extra_info, on="player_id", how="left")

missing = bayer_squad_df.loc[bayer_squad_df["photo_url"].isna(), "player_name"].tolist()
assert not missing, f"Add these players to player_extra_info.csv: {missing}"

export(bayer_squad_df, "bayer_squad.csv")
bayer_squad_df

bayer_squad.csv: 24 rows


,player_id,player_name,jersey_number,country,common_name,jersey_name,position,photo_url
0,3500,Granit Xhaka,34,Switzerland,Granit Xhaka,XHAKA,Midfielder,https://img.sofascore.com/api/v1/player/117777...
1,8221,Jonathan Tah,4,Germany,Jonathan Tah,TAH,Defender,https://img.sofascore.com/api/v1/player/227672...
2,8403,Nadiem Amiri,11,Germany,Nadiem Amiri,AMIRI,Midfielder,https://img.sofascore.com/api/v1/player/327755...
3,8667,Lukáš Hrádecký,1,Finland,Lukáš Hrádecký,HRÁDECKÝ,Goalkeeper,https://img.sofascore.com/api/v1/player/15219/...
4,8804,Jonas Hofmann,7,Germany,Jonas Hofmann,HOFMANN,Midfielder,https://img.sofascore.com/api/v1/player/142390...
5,9195,Robert Andrich,8,Germany,Robert Andrich,ANDRICH,Midfielder,https://img.sofascore.com/api/v1/player/168903...
6,10336,Alejandro Grimaldo García,20,Spain,Alejandro Grimaldo,GRIMALDO,Defender,https://img.sofascore.com/api/v1/player/177177...
7,27133,Odilon Kossonou,6,Côte d'Ivoire,Odilon Kossonou,KOSSONOU,Defender,https://img.sofascore.com/api/v1/player/973437...
8,28239,Adam Hložek,23,Czech Republic,Adam Hložek,HLOŽEK,Forward,https://img.sofascore.com/api/v1/player/963801...
9,28268,Exequiel Alejandro Palacios,25,Argentina,Exequiel Palacios,PALACIOS,Midfielder,https://img.sofascore.com/api/v1/player/822600...


## Passes

In [5]:
bayer_passes_df = bayer_events_df[bayer_events_df["type"] == "Pass"].copy()

# A pass is successful when StatsBomb records no outcome
bayer_passes_df["pass_result"] = bayer_passes_df["pass_outcome"].isna().map({True: "Success", False: "Failed"})
# "Success" = the pass led to a goal (label kept as is for the Power BI measures)
bayer_passes_df["pass_goal_assist"] = (bayer_passes_df["pass_goal_assist"] == True).map({True: "Success", False: "Failed"})
# Always empty for passes; kept only so the CSV schema used by Power BI does not change
bayer_passes_df["shot_outcome"] = None

bayer_passes_df = bayer_passes_df[[
    "id", "match_id", "period", "minute", "player", "player_id", "position",
    "location_x", "location_y", "pass_recipient", "pass_recipient_id",
    "pass_end_location_x", "pass_end_location_y", "pass_type", "pass_technique",
    "pass_height", "pass_length", "pass_outcome", "pass_result", "shot_outcome",
    "pass_goal_assist", "team",
]]

export(bayer_passes_df, "passes_bayer.csv")
print(f"Pass accuracy: {(bayer_passes_df['pass_result'] == 'Success').mean():.1%}")

passes_bayer.csv: 24244 rows
Pass accuracy: 88.1%


## Shots

Note: own goals scored by opponents are not `Shot` events, so goals counted from this table
(86) are slightly lower than Bayer's real goal tally (89).

In [6]:
bayer_shots_df = bayer_events_df.loc[bayer_events_df["type"] == "Shot", [
    "id", "match_id", "period", "minute", "player", "player_id", "position",
    "location_x", "location_y", "shot_end_location_x", "shot_end_location_y",
    "shot_type", "shot_technique", "shot_body_part", "shot_outcome",
    "shot_statsbomb_xg", "under_pressure", "team",
]]

export(bayer_shots_df, "shots_bayer.csv")
print(f"Goals from shots: {(bayer_shots_df['shot_outcome'] == 'Goal').sum()}, "
      f"xG: {bayer_shots_df['shot_statsbomb_xg'].sum():.1f}")

shots_bayer.csv: 623 rows
Goals from shots: 86, xG: 72.7


## Lineup positions

Each lineup row holds a list of position spells (`from` / `to` match clock, e.g. `"63:12"`).
Unused substitutes have an **empty list**: they are dropped here, otherwise they would be counted
as having played.

In [7]:
spells = lineups_df.explode("positions", ignore_index=True)
spells = spells[spells["positions"].notna()].reset_index(drop=True)   # drops unused substitutes
spells = pd.concat(
    [spells.drop(columns="positions"), pd.json_normalize(spells["positions"].tolist())],
    axis=1,
)
spells[["player_name", "match_id", "position", "from", "to", "start_reason", "end_reason"]].head()

,player_name,match_id,position,from,to,start_reason,end_reason
0,Granit Xhaka,3895052,Left Defensive Midfield,00:00,74:36,Starting XI,Tactical Shift
1,Granit Xhaka,3895052,Left Center Midfield,74:36,NaN,Tactical Shift,Final Whistle
2,Jonathan Tah,3895052,Center Back,00:00,NaN,Starting XI,Final Whistle
3,Nadiem Amiri,3895052,Left Center Forward,88:31,NaN,Substitution - On (Tactical),Final Whistle
4,Lukáš Hrádecký,3895052,Goalkeeper,00:00,NaN,Starting XI,Final Whistle


## Most used starting XI

In [8]:
starters = spells[spells["start_reason"] == "Starting XI"]

total_starts = starters.groupby("player_id").size().rename("starts").reset_index()

main_position = (
    starters.groupby(["player_id", "position"]).size().rename("n").reset_index()
    .sort_values("n", ascending=False)
    .drop_duplicates("player_id")
    .rename(columns={"position": "main_position"})
    [["player_id", "main_position"]]
)

player_info = lineups_df.drop_duplicates("player_id")[["player_id", "player_name", "jersey_number", "country"]]

xi_df = (
    total_starts.merge(player_info, on="player_id").merge(main_position, on="player_id")
    .sort_values("starts", ascending=False)
    .head(11)
    [["player_id", "jersey_number", "player_name", "country", "main_position", "starts"]]
)

# Pitch coordinates (0-100) for the Power BI lineup visual — Xabi Alonso's 3-4-2-1
position_coords = {
    "Goalkeeper": (50, 5),
    "Left Center Back": (30, 20), "Center Back": (50, 20), "Right Center Back": (70, 20),
    "Left Wing Back": (15, 35), "Right Wing Back": (85, 35),
    "Left Defensive Midfield": (35, 33), "Right Defensive Midfield": (65, 33),
    "Left Attacking Midfield": (35, 45), "Right Attacking Midfield": (65, 45),
    "Center Forward": (50, 55),
}
xi_df["x"] = xi_df["main_position"].map(lambda p: position_coords[p][0])
xi_df["y"] = xi_df["main_position"].map(lambda p: position_coords[p][1])

# Sanity check: one player per position, otherwise the visual would stack two players
assert xi_df["main_position"].is_unique, xi_df["main_position"].value_counts()

export(xi_df, "starting_bayer.csv")
xi_df

starting_bayer.csv: 11 rows


,player_id,jersey_number,player_name,country,main_position,starts,x,y
4,8667,1,Lukáš Hrádecký,Finland,Goalkeeper,33,50,5
0,3500,34,Granit Xhaka,Switzerland,Right Defensive Midfield,32,65,33
7,10336,20,Alejandro Grimaldo García,Spain,Left Wing Back,31,15,35
2,8221,4,Jonathan Tah,Germany,Center Back,30,50,20
14,32712,30,Jeremie Frimpong,Netherlands,Right Wing Back,27,85,35
5,8804,7,Jonas Hofmann,Germany,Right Attacking Midfield,26,65,45
17,40724,10,Florian Wirtz,Germany,Left Attacking Midfield,26,35,45
12,30606,12,Edmond Fayçal Tapsoba,Burkina Faso,Left Center Back,23,30,20
11,28268,25,Exequiel Alejandro Palacios,Argentina,Left Defensive Midfield,21,35,33
9,27133,6,Odilon Kossonou,Côte d'Ivoire,Right Center Back,21,70,20


## Minutes played

A player still on the pitch at the final whistle has no `to` time: we use the actual end of
the match (last event of the second half, stoppage time included) instead of a flat 90:00.

In [9]:
def clock_to_seconds(t):
    m, s = map(int, t.split(":"))
    return m * 60 + s

# End of each match = clock of its last event
match_end_sec = (
    events_df.assign(clock=events_df["minute"] * 60 + events_df["second"])
    .groupby("match_id")["clock"].max()
)

spells["from_sec"] = spells["from"].apply(clock_to_seconds)
spells["to_sec"] = [
    clock_to_seconds(t) if isinstance(t, str) else match_end_sec[m]
    for t, m in zip(spells["to"], spells["match_id"])
]
spells["duration_min"] = (spells["to_sec"] - spells["from_sec"]) / 60
assert (spells["duration_min"] >= 0).all()

minutes_df = (
    spells.groupby(["match_id", "player_id", "player_name"], as_index=False)["duration_min"].sum()
    .rename(columns={"duration_min": "minutes_played"})
)
minutes_df["minutes_played"] = minutes_df["minutes_played"].round(2)

export(minutes_df, "minutes_bayer.csv")

minutes_bayer.csv: 516 rows


In [10]:
# Sanity checks: 11 players on the pitch at all times (fewer after a red card)
per_match = minutes_df.groupby("match_id").agg(players=("player_id", "size"), total=("minutes_played", "sum"))
per_match["avg_on_pitch"] = per_match["total"] / (match_end_sec[per_match.index] / 60)
per_match.describe().round(2)

,players,total,avg_on_pitch
count,34.00,34.00,34.00
mean,15.18,1041.00,11.00
std,1.09,21.35,0.01
min,12.00,988.02,10.96
25%,15.00,1033.70,10.99
50%,16.00,1037.95,11.00
75%,16.00,1053.68,11.00
max,16.00,1086.80,11.00


In [11]:
minutes_df.groupby("player_name")["minutes_played"].sum().sort_values(ascending=False).round(0)

player_name
Lukáš Hrádecký                    3124.0
Granit Xhaka                      2947.0
Alejandro Grimaldo García         2910.0
Jonathan Tah                      2761.0
Florian Wirtz                     2456.0
Jeremie Frimpong                  2305.0
Jonas Hofmann                     2266.0
Edmond Fayçal Tapsoba             2201.0
Odilon Kossonou                   1900.0
Exequiel Alejandro Palacios       1897.0
Robert Andrich                    1801.0
Victor Okoh Boniface              1601.0
Piero Martín Hincapié Reyna       1580.0
Josip Stanišić                    1348.0
Patrik Schick                     1118.0
Amine Adli                         972.0
Nathan Tella                       894.0
Adam Hložek                        538.0
Borja Iglesias Quintas             275.0
Arthur Augusto de Matos Soares     198.0
Nadiem Amiri                       114.0
Matěj Kovář                         94.0
Gustavo Adolfo Puerta Molano        71.0
Noah Mbamba                         23.0
Name

Further transformations (measures, per-90 stats, visuals) are done in Power BI.